# Project 1 - DS 2002
### Max St. Clair

#### Import libraries

In [1]:
import os
import numpy
import numpy
import datetime
import certifi
import pandas as pd
import pymongo
import sqlalchemy
from sqlalchemy import create_engine, text

#### Declare & Assign Connection Variables for the MySQL Server & Databases

In [2]:
host_name = "localhost"
port = "3306"
user_id = "root"
pwd = "Major2014"

src_dbname = "sakila"
dst_dbname = "sakila_dw"

#### Define Functions for Getting Data From and Setting Data Into Databases

In [5]:
def get_dataframe(user_id, pwd, host_name, db_name, sql_query):
    conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}/{db_name}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    dframe = pd.read_sql(sql_query, connection);
    connection.close()
    
    return dframe


def set_dataframe(user_id, pwd, host_name, db_name, df, table_name, pk_column, db_operation):
    conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}/{db_name}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    
    if db_operation == "insert":
        df.to_sql(table_name, con=connection, index=False, if_exists='replace')
        connection.execute(text(f"ALTER TABLE {table_name} ADD PRIMARY KEY ({pk_column});"))
            
    elif db_operation == "update":
        df.to_sql(table_name, con=connection, index=False, if_exists='append')
    
    connection.close()

#### Create New Data Warehouse Dataframe

In [7]:
conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}"
sqlEngine = create_engine(conn_str, pool_recycle=3600)
connection = sqlEngine.connect()

connection.execute(text(f"DROP DATABASE IF EXISTS `{dst_dbname}`;"))
connection.execute(text(f"CREATE DATABASE `{dst_dbname}`;"))
connection.execute(text(f"USE {dst_dbname};"))

connection.close()

#### Extract data from SQL warehouse

In [37]:
sql_customers = "SELECT * FROM sakila.customer;"
df_customers = get_dataframe(user_id, pwd, host_name, src_dbname, sql_customers)
df_customers.head(2)

,customer_id,store_id,first_name,last_name,email,address_id,active,create_date,last_update
0,1,1,MARY,SMITH,MARY.SMITH@sakilacustomer.org,5,1,2006-02-14 22:04:36,2006-02-15 04:57:20
1,2,1,PATRICIA,JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,6,1,2006-02-14 22:04:36,2006-02-15 04:57:20


In [47]:
sql_store = "SELECT * FROM sakila.store;"
df_store = get_dataframe(user_id, pwd, host_name, src_dbname, sql_store)
df_store.head()

,store_id,manager_staff_id,address_id,last_update
0,1,1,1,2006-02-15 04:57:12
1,2,2,2,2006-02-15 04:57:12


In [59]:
sql_film = "SELECT * FROM sakila.film;"
df_film = get_dataframe(user_id, pwd, host_name, src_dbname, sql_film)
df_film.head(2)

,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,None,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2006-02-15 05:03:42
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,1,None,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2006-02-15 05:03:42


Import a location dimension by joining address with city and country tables

In [21]:
sql_locations = """
    SELECT
        a.address_id AS address_id,
        a.address,
        a.address2,
        a.district,
        a.postal_code,
        a.phone,
        c.city AS city_name,
        co.country AS country_name
    FROM
        address a
    JOIN
        city c ON a.city_id = c.city_id
    JOIN
        country co ON c.country_id = co.country_id;
"""
dim_locations = get_dataframe(user_id, pwd, host_name, src_dbname, sql_locations)
dim_locations.head(2)

,address_id,address,address2,district,postal_code,phone,city_name,country_name
0,222,1168 Najafabad Parkway,,Kabol,40301,886649065861,Kabul,Afghanistan
1,446,1924 Shimonoseki Drive,,Batna,52625,406784385440,Batna,Algeria


#### Declare & Assign Connection Variables for the MongoDB Server, the MySQL Server & Database

In [25]:
mongodb_args = {
    "user_name" : "MaxStClair",
    "password" : "Bluedog$2002",
    "cluster_name" : "Cluster0",
    "cluster_subnet" : "hatl8",
    "cluster_location" : "atlas", # "local"
    "db_name" : "sakila_dw"
}

#### Define Functions for Getting Data From and Setting Data Into Databases

In [27]:
def get_mongo_client(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the cluster_location parameter.")
    
    else:
        if args["cluster_location"] == "atlas":
            connect_str = f"mongodb+srv://{args['user_name']}:{args['password']}@"
            connect_str += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net"
            client = pymongo.MongoClient(connect_str, tlsCAFile=certifi.where())
            
        elif args["cluster_location"] == "local":
            client = pymongo.MongoClient("mongodb://localhost:27017/")
        
    return client


def get_mongo_dataframe(mongo_client, db_name, collection, query):
    '''Query MongoDB, and fill a python list with documents to create a DataFrame'''
    db = mongo_client[db_name]
    dframe = pd.DataFrame(list(db[collection].find(query)))
    dframe.drop(['_id'], axis=1, inplace=True)
    mongo_client.close()
    
    return dframe


def set_mongo_collections(mongo_client, db_name, data_directory, json_files):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()

#### Populate MongoDB with Source Data

In [31]:
client = get_mongo_client(**mongodb_args)

# Gets the path of the Current Working Directory for this Notebook,
# and then Appends the 'data' directory.
data_dir = os.path.join(os.getcwd(), 'DS2002_Project_1')

json_files = {"inventory" : "dim_inventory.json"}

set_mongo_collections(client, mongodb_args["db_name"], data_dir, json_files)

#### Extract the inventory table through MongoDB

In [33]:
client = get_mongo_client(**mongodb_args)

query = {} # Select all elements (columns), and all documents (rows).
collection = "inventory"

dim_inventory = get_mongo_dataframe(client, mongodb_args["db_name"], collection, query)
dim_inventory.head(2)

,inventory_id,film_id,store_id,last_update
0,1,1,1,2006-02-15 05:09:17
1,2,1,1,2006-02-15 05:09:17


#### Import the staff table, which comes from a CSV

In [43]:
df_staff = pd.read_csv('DS2002_Project_1/df_staff.csv')
df_staff.head() 

,staff_id,first_name,last_name,email,store_id,username,password
0,1,Mike,Hillyer,Mike.Hillyer@sakilastaff.com,1,Mike,8cb2237d0679ca88db6464eac60da96345513964
1,2,Jon,Stephens,Jon.Stephens@sakilastaff.com,2,Jon,NaN


#### Import the rental table, which is the fact table for this database

In [53]:
sql_rental = "SELECT * FROM sakila.rental;"
df_rental = get_dataframe(user_id, pwd, host_name, src_dbname, sql_rental)
df_rental.head(2)

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53


#### Create rental_film_detail, by joining film with inventory then with rental fact table

In [63]:
# Join rental and inventory on inventory_id
rental_inventory_df = df_rental.merge(dim_inventory, on='inventory_id', how='inner')

# Then join the result with film on film_id
rental_film_detail_df = rental_inventory_df.merge(df_film, on='film_id', how='inner')

# Select essential columns
rental_film_detail_df = rental_film_detail_df[[
    'rental_id', 'rental_date', 'return_date', 'film_id', 
    'title', 'description', 'release_year', 'length', 'rating'
]]
rental_film_detail_df.head(2)

,rental_id,rental_date,return_date,film_id,title,description,release_year,length,rating
0,1,2005-05-24 22:53:30,2005-05-26 22:04:30,80,BLANKET BEVERLY,A Emotional Documentary of a Student And a Gir...,2006,148,G
1,16,2005-05-25 00:43:11,2005-05-26 04:42:11,86,BOOGIE AMELIE,A Lacklusture Character Study of a Husband And...,2006,121,R


#### Create rental_staff_detail

In [65]:
rental_staff_detail = df_rental.merge(df_staff, on='staff_id', how='inner')

In [67]:
rental_staff_detail.head(2)

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update,first_name,last_name,email,store_id,username,password
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53,Mike,Hillyer,Mike.Hillyer@sakilastaff.com,1,Mike,8cb2237d0679ca88db6464eac60da96345513964
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53,Mike,Hillyer,Mike.Hillyer@sakilastaff.com,1,Mike,8cb2237d0679ca88db6464eac60da96345513964


#### Write the fact tables back to the sakila_dw database

In [71]:
table_name = "rental_film_detail_df"
primary_key = "rental_id"
db_operation = "insert"

set_dataframe(user_id, pwd, host_name, dst_dbname, rental_film_detail_df, table_name, primary_key, db_operation)

In [74]:
table_name = "rental_staff_detail"
primary_key = "rental_id"
db_operation = "insert"

set_dataframe(user_id, pwd, host_name, dst_dbname, rental_staff_detail, table_name, primary_key, db_operation)

#### Demonstrate that the new data warehouse exists and contains the correct data

I wanted to see which films were rented out most frequently, so this SQL statement returns the title of each film and how many times it's been rented out. I also wanted to see who the best employees are, so the second statement returns rental by employee

In [86]:
sql_test = """
    SELECT
        title,
        COUNT(rental_id) AS rental_count
    FROM
        rental_film_detail_df
    GROUP BY
        title
    ORDER BY
        rental_count DESC;
""".format(dst_dbname)

df_test = get_dataframe(user_id, pwd, host_name, dst_dbname, sql_test)
df_test.head()

,title,rental_count
0,BUCKET BROTHERHOOD,34
1,APACHE DIVINE,31
2,BUTTERFLY CHOCOLAT,30
3,CAT CONEHEADS,30
4,BOOGIE AMELIE,29


In [90]:
sql_test2 = """
    SELECT
        first_name,
        last_name,
        COUNT(rental_id) AS rental_count
    FROM
        rental_staff_detail
    GROUP BY
        first_name,
        last_name
    ORDER BY
        rental_count DESC;
""".format(dst_dbname)

df_test2 = get_dataframe(user_id, pwd, host_name, dst_dbname, sql_test2)
df_test2.head()

,first_name,last_name,rental_count
0,Mike,Hillyer,8040
1,Jon,Stephens,8004
